# 11. Citation and Evidence Coverage

Grounded answers are better when they expose their evidence. This notebook builds citation-analysis helpers that measure whether claims are cited completely, mapped to the right sources, and missing important documents.

## Learning goals

- parse citations from answer text
- map each claim to likely supporting sources
- score citation completeness and evidence coverage
- detect missing or spurious citations with transparent heuristics

## Why citations deserve their own eval

Faithfulness asks whether the answer stayed inside the evidence. Citation quality asks whether the answer showed its work. A system can be supported yet still hard to audit if citations are incomplete or wrong.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import faiss
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def embed_texts(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    return embed_model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def build_faiss_index(texts):
    texts = list(texts)
    embeddings = embed_texts(texts).astype(np.float32)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return {
        "texts": texts,
        "embeddings": embeddings,
        "index": index,
    }

def search_index(query, store, k=5):
    k = min(k, len(store["texts"]))
    if k <= 0:
        return []

    query_vector = embed_texts([query]).astype(np.float32)
    scores, indices = store["index"].search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append(
            {
                "index": int(idx),
                "score": float(score),
                "text": store["texts"][idx],
            }
        )
    return results

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Utility helpers

We will work claim by claim again, but now every claim also carries explicit citation tags like [D5] or [D6, D1].

In [ ]:
import re

random.seed(13)

STOPWORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "with", "by",
    "at", "from", "during", "into", "over", "under", "after", "before", "than",
    "what", "which", "where", "when", "how", "did", "does", "is", "are", "was",
    "were", "be", "been", "it", "its", "their", "that", "this", "these", "those"
}


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def content_words(text):
    return [token for token in normalize(text).split() if token not in STOPWORDS]


def split_sentences(text):
    cleaned = text.replace("\n", " ").strip()
    parts = re.split(r"(?<=[.!?])\s+", cleaned)
    sentences = []
    for part in parts:
        for subpart in part.split(";"):
            subpart = subpart.strip()
            if subpart:
                sentences.append(subpart)
    return sentences


def extract_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text))


def clip(text, width=82):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


def overlap_score(claim, evidence):
    claim_terms = set(content_words(claim))
    evidence_terms = set(content_words(evidence))
    if not claim_terms:
        return 0.0
    lexical = len(claim_terms & evidence_terms) / len(claim_terms)
    claim_numbers = extract_numbers(claim)
    if claim_numbers:
        numeric = len(claim_numbers & extract_numbers(evidence)) / len(claim_numbers)
    else:
        numeric = 0.0
    return 0.75 * lexical + 0.25 * numeric

## Step 2 — Recreate the local corpus with sentence-level passages

Citation analysis works best when source units are small and identifiable. We will tag every sentence with a document id and passage id.

In [ ]:
CORPUS = [
    {
        "doc_id": "D1",
        "title": "Harbor District Microgrid Pilot",
        "text": "The Harbor district launched a resilience pilot in 2024. It installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery. During two grid outages, the microgrid kept the health clinic and water pumps online for 11 hours. Operators reported a 34 percent reduction in diesel generator use compared with the previous summer.",
    },
    {
        "doc_id": "D2",
        "title": "Cedar Wastewater Heat Recovery Memo",
        "text": "The Cedar treatment plant now captures heat from outgoing effluent. The recovered heat keeps digesters warm and supplies a nearby greenhouse loop. Plant managers reported an 18 percent reduction in natural gas consumption. The memo estimates a payback period of about 5.5 years.",
    },
    {
        "doc_id": "D3",
        "title": "Northwind Offshore Wind Siting Brief",
        "text": "The Northwind team shifted turbine placement 9 kilometers east to avoid a major bird migration corridor. The export cable shares an existing shipping channel for the first 14 kilometers. Environmental review says construction noise is the largest short-term risk. The project brief estimates annual output could power 210000 homes.",
    },
    {
        "doc_id": "D4",
        "title": "Library Retrofit Case Study",
        "text": "A central library replaced chillers with ground-source heat pumps and smart ventilation controls. Energy use intensity fell from 212 to 148 kilowatt-hours per square meter. Summer comfort complaints dropped by 40 percent. Total project cost was 4.2 million dollars with an 11-year payback.",
    },
    {
        "doc_id": "D5",
        "title": "Drought Response Handbook",
        "text": "Stage two restrictions begin when reservoir storage stays below 62 percent for 21 consecutive days. The handbook says leak repairs can save up to 12 percent of summer demand. Recharge basins should rest every seventh day to prevent compaction. Public messaging focuses on irrigation schedules and the repair hotline.",
    },
    {
        "doc_id": "D6",
        "title": "Transit Electrification Plan",
        "text": "Phase one purchases 18 battery buses by 2026. The full program targets 42 electric buses by 2028. Overnight depot charging serves most vehicles, while an on-route pantograph at Central Station handles the peak corridor. The plan forecasts 22 percent lower maintenance cost than diesel operations.",
    },
    {
        "doc_id": "D7",
        "title": "Estuary Flood Barrier Memo",
        "text": "The estuary barrier closes only during storm-surge warnings above 1.7 meters. Wetlands restoration is expected to delay peak flooding by 35 minutes. The annual maintenance budget is 8.4 million dollars. Fishing access is preserved through a side channel.",
    },
    {
        "doc_id": "D8",
        "title": "Data Center Water Reuse Project",
        "text": "A reclaim system now reuses 68 percent of cooling water at the municipal data center. The facility pairs with treated greywater from the wastewater plant. Summer potable water demand fell by 41 percent after deployment. Filtration membranes are replaced every 18 months.",
    },
]


def make_passages(corpus):
    passages = []
    for doc in corpus:
        for idx, sentence in enumerate(split_sentences(doc["text"]), start=1):
            passages.append(
                {
                    "passage_id": "{}-P{}".format(doc["doc_id"], idx),
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "text": sentence,
                }
            )
    return passages


PASSAGES = make_passages(CORPUS)
PASSAGE_STORE = build_faiss_index([passage["title"] + ". " + passage["text"] for passage in PASSAGES])

print_rows(PASSAGES[:10], columns=["passage_id", "doc_id", "text"])

## Step 3 — Create a few cited-answer examples

Each example includes the answer text plus the documents that should support each claim. That gives us a tiny local benchmark for citation completeness.

In [ ]:
CITATION_CASES = [
    {
        "name": "complete_multi_source",
        "question": "Summarize the drought trigger and the water reuse project's potable water impact.",
        "answer": "Stage two drought rules begin when reservoir storage stays below 62 percent for 21 days [D5]. The water reuse project reduced summer potable water demand by 41 percent [D8].",
        "expected_docs": [["D5"], ["D8"]],
    },
    {
        "name": "missing_source",
        "question": "Summarize the drought trigger and the water reuse project's potable water impact.",
        "answer": "Stage two drought rules begin when reservoir storage stays below 62 percent for 21 days [D5]. The water reuse project reduced summer potable water demand by 41 percent.",
        "expected_docs": [["D5"], ["D8"]],
    },
    {
        "name": "wrong_source",
        "question": "What ecological change was made in the wind siting brief and what output was projected?",
        "answer": "The wind team shifted turbines 9 kilometers east to avoid the migration corridor [D7]. Annual output was projected to power 210000 homes [D3].",
        "expected_docs": [["D3"], ["D3"]],
    },
    {
        "name": "bundled_claim",
        "question": "How do the transit and microgrid projects quantify savings?",
        "answer": "The transit plan expects 22 percent lower maintenance cost and the Harbor microgrid reported 34 percent less diesel generator use [D6, D1].",
        "expected_docs": [["D6", "D1"]],
    },
]

print_rows(
    [{"name": case["name"], "answer": clip(case["answer"], 96)} for case in CITATION_CASES],
    columns=["name", "answer"],
)

## Step 4 — Parse claims and citations

Before scoring anything, make the structure explicit. Each sentence becomes a claim plus a list of cited document ids.

In [ ]:
def extract_citation_ids(text):
    groups = re.findall(r"\[([A-Z0-9,\s]+)\]", text)
    doc_ids = []
    for group in groups:
        for part in group.split(","):
            part = part.strip()
            if part:
                doc_ids.append(part)
    return sorted(set(doc_ids))


def strip_citations(text):
    return re.sub(r"\[[^\]]+\]", "", text).strip()


def parse_cited_claims(answer):
    claims = []
    for sentence in split_sentences(answer):
        cleaned = strip_citations(sentence)
        if cleaned:
            claims.append(
                {
                    "claim_text": cleaned,
                    "citations": extract_citation_ids(sentence),
                }
            )
    return claims


parsed_demo = parse_cited_claims(CITATION_CASES[0]["answer"])
print_rows(parsed_demo, columns=["claim_text", "citations"])

## Step 5 — Map claims to likely supporting sources

We can retrieve candidate passages for each claim and aggregate them into a lightweight source map. This helps us check whether a citation points to the document that actually supports the claim.

In [ ]:
def map_claim_to_sources(claim, k=4):
    results = search_index(claim, PASSAGE_STORE, k=k)
    mapped = []
    for result in results:
        passage = PASSAGES[result["index"]]
        mapped.append(
            {
                "doc_id": passage["doc_id"],
                "passage_id": passage["passage_id"],
                "dense_score": round(result["score"], 4),
                "support_score": round(overlap_score(claim, passage["text"]), 3),
                "text": clip(passage["text"], 78),
            }
        )
    return sorted(mapped, key=lambda row: (row["support_score"], row["dense_score"]), reverse=True)


for case in CITATION_CASES[:2]:
    parsed_claims = parse_cited_claims(case["answer"])
    print("\nCase:", case["name"])
    for parsed in parsed_claims:
        print("Claim:", parsed["claim_text"])
        print_rows(map_claim_to_sources(parsed["claim_text"], k=3), columns=["doc_id", "passage_id", "support_score", "text"])
        print()

## Step 6 — Score citation completeness and coverage

We will track four simple outputs:

- cited_claim_ratio: how many claims cite anything at all?
- citation_completeness: how many claims cite at least one correct supporting document?
- evidence_coverage: how much of the required document set appears in the citations?
- missing_source_count: how many claims left out a source they should have cited?

In [ ]:
def score_citation_case(case, threshold=0.55):
    parsed_claims = parse_cited_claims(case["answer"])
    diagnostics = []
    for claim_idx, (parsed, expected_docs) in enumerate(zip(parsed_claims, case["expected_docs"]), start=1):
        expected_set = set(expected_docs)
        citations = set(parsed["citations"])
        mapped = map_claim_to_sources(parsed["claim_text"], k=4)
        supported_docs = []
        for row in mapped:
            if row["support_score"] >= threshold and row["doc_id"] not in supported_docs:
                supported_docs.append(row["doc_id"])
        diagnostics.append(
            {
                "claim_index": claim_idx,
                "claim": clip(parsed["claim_text"], 72),
                "citations": sorted(citations),
                "expected_docs": sorted(expected_set),
                "correct": bool(citations & expected_set),
                "missing_docs": sorted(expected_set - citations),
                "spurious_docs": sorted(citations - expected_set),
                "suggested_docs": supported_docs[:3],
            }
        )

    claim_count = len(diagnostics) if diagnostics else 1
    expected_union = set(doc for doc_list in case["expected_docs"] for doc in doc_list)
    cited_claim_ratio = statistics.mean(1.0 if row["citations"] else 0.0 for row in diagnostics) if diagnostics else 0.0
    citation_completeness = statistics.mean(1.0 if row["correct"] else 0.0 for row in diagnostics) if diagnostics else 0.0
    correct_union = set()
    for row in diagnostics:
        correct_union.update(set(row["citations"]) & set(row["expected_docs"]))
    evidence_coverage = len(correct_union) / len(expected_union) if expected_union else 0.0

    return {
        "claim_count": len(diagnostics),
        "cited_claim_ratio": round(cited_claim_ratio, 3),
        "citation_completeness": round(citation_completeness, 3),
        "evidence_coverage": round(evidence_coverage, 3),
        "missing_source_count": sum(1 for row in diagnostics if row["missing_docs"]),
        "spurious_citation_count": sum(len(row["spurious_docs"]) for row in diagnostics),
        "diagnostics": diagnostics,
    }


for case in CITATION_CASES[:2]:
    scored = score_citation_case(case)
    print("\nCase:", case["name"])
    print_rows(scored["diagnostics"], columns=["claim_index", "claim", "citations", "expected_docs", "correct", "missing_docs", "suggested_docs"])
    print("Summary:", {key: value for key, value in scored.items() if key != "diagnostics"})

## Step 7 — Run the full citation benchmark

Because the benchmark is tiny and local, you can inspect every line. That makes it a good regression harness when you iterate on prompts or citation formatting policies later.

In [ ]:
benchmark_rows = []
for case in CITATION_CASES:
    scored = score_citation_case(case)
    benchmark_rows.append(
        {
            "name": case["name"],
            "claim_count": scored["claim_count"],
            "cited_claim_ratio": scored["cited_claim_ratio"],
            "citation_completeness": scored["citation_completeness"],
            "evidence_coverage": scored["evidence_coverage"],
            "missing_source_count": scored["missing_source_count"],
            "spurious_citation_count": scored["spurious_citation_count"],
        }
    )

print_rows(
    benchmark_rows,
    columns=[
        "name",
        "claim_count",
        "cited_claim_ratio",
        "citation_completeness",
        "evidence_coverage",
        "missing_source_count",
        "spurious_citation_count",
    ],
)

## Step 8 — Suggest missing sources automatically

A useful citation evaluator should not only fail an answer, but also point toward the missing evidence. Here we use the claim-to-source mapper as a cheap source-suggestion tool.

In [ ]:
def suggest_missing_sources(case, threshold=0.55):
    scored = score_citation_case(case, threshold=threshold)
    suggestions = []
    for row in scored["diagnostics"]:
        for missing_doc in row["missing_docs"]:
            suggestions.append(
                {
                    "claim": row["claim"],
                    "missing_doc": missing_doc,
                    "suggested_docs": row["suggested_docs"],
                }
            )
    return suggestions


missing_case = CITATION_CASES[1]
print("Case:", missing_case["name"])
print_rows(suggest_missing_sources(missing_case), columns=["claim", "missing_doc", "suggested_docs"])

## Step 9 — Inspect the weakest citation case in detail

This is the kind of table you would keep for debugging. It shows each claim, the cited docs, the docs that should have been cited, and the docs the evidence search thinks are most plausible.

In [ ]:
weakest_case = min(CITATION_CASES, key=lambda case: score_citation_case(case)["citation_completeness"])
weakest_score = score_citation_case(weakest_case)

print("Weakest case:", weakest_case["name"])
print(weakest_case["answer"])
print()
print_rows(
    weakest_score["diagnostics"],
    columns=["claim_index", "claim", "citations", "expected_docs", "missing_docs", "spurious_docs", "suggested_docs"],
)

## Step 10 — Small repair checklist

When citation quality is weak, the fix is usually one of four things:

- ask the model to cite every factual sentence
- require document ids in a strict format
- increase retrieval recall so the needed source is available
- rerank or filter context so the generator sees less distracting evidence

In [ ]:
repair_rows = []
for case in CITATION_CASES:
    scored = score_citation_case(case)
    repair_rows.append(
        {
            "name": case["name"],
            "needs_more_citations": scored["cited_claim_ratio"] < 1.0,
            "needs_better_source_matching": scored["citation_completeness"] < 1.0,
            "needs_more_evidence_coverage": scored["evidence_coverage"] < 1.0,
        }
    )

print_rows(repair_rows, columns=["name", "needs_more_citations", "needs_better_source_matching", "needs_more_evidence_coverage"])

## Recap

You now have local helpers for parsing citations, mapping claims to evidence, scoring completeness, and suggesting missing sources. The final notebook in this module uses those ideas in a full RAG ablation lab.